# 03 — Transformer Architecture

Define a minimal decoder-only (GPT-style) transformer for next-token
prediction over the 3-state alphabet.  Walk through the architecture,
count parameters, and verify the forward pass.

In [1]:
import torch

## Model

Architecture: token embedding → learned positional embedding →
N causal TransformerEncoder layers (pre-norm) → linear head over vocab.

This is a standard GPT-style setup.  The causal attention mask ensures
position *t* only attends to positions 0 … t, so the model predicts the
next token from its causal history — the same information used to define
the Markov chain.

In [2]:
import sys
from pathlib import Path

# Resolve project root and add project-local packages/ to sys.path.
cwd = Path.cwd().resolve()
project_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "packages" / "pytorch_models" / "markov_transformer.py").exists()
    ),
    None,
)
if project_root is None:
    alt_root = cwd / "projects" / "markov-chain-learning"
    if (alt_root / "packages" / "pytorch_models" / "markov_transformer.py").exists():
        project_root = alt_root

if project_root is None:
    raise RuntimeError(
        "Could not locate markov-chain-learning project root from current working directory"
    )

packages_dir = project_root / "packages"
if str(packages_dir) not in sys.path:
    sys.path.insert(0, str(packages_dir))

from pytorch_models import MarkovTransformer

## Instantiate & inspect

In [4]:
# Default config — kept small so the experiment runs on CPU
VOCAB_SIZE = 3
D_MODEL = 3 * 2
MAX_LEN = 32

model = MarkovTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    max_len=MAX_LEN,
)
print(model)

MarkovTransformer(
  (input_proj): Linear(in_features=3, out_features=6, bias=False)
  (rel_pos_bias): Embedding(32, 1)
  (layers): ModuleList(
    (0-1): 2 x Attention(
      (W_q): Linear(in_features=6, out_features=6, bias=False)
      (W_k): Linear(in_features=6, out_features=6, bias=False)
      (W_v): Linear(in_features=6, out_features=6, bias=False)
      (W_o): Linear(in_features=6, out_features=6, bias=False)
    )
  )
  (head): Linear(in_features=6, out_features=3, bias=False)
)


In [5]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")

# Per-module breakdown
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20} {n:>6,} params")

Total parameters    : 356
Trainable parameters: 356
  input_proj               18 params
  rel_pos_bias             32 params
  layers                  288 params
  head                     18 params


## Smoke test

In [6]:
# Verify output shapes for a random batch
batch_size, seq_len = 4, 16
x = torch.randint(0, VOCAB_SIZE, (batch_size, seq_len))
logits = model(x)
print(f"Input  shape: {x.shape}")
print(f"Output shape: {logits.shape}  (expected: {(batch_size, seq_len, VOCAB_SIZE)})")
assert logits.shape == (batch_size, seq_len, VOCAB_SIZE)

# Softmax probabilities at position 0 should sum to 1 for each batch element
probs0 = logits[:, 0, :].softmax(dim=-1)
assert torch.allclose(probs0.sum(dim=-1), torch.ones(batch_size), atol=1e-5)
print("✓ Output shapes and probability sums are correct")

Input  shape: torch.Size([4, 16])
Output shape: torch.Size([4, 16, 3])  (expected: (4, 16, 3))
✓ Output shapes and probability sums are correct
